# What a miss costs, without guesses

**Started with `notebooks/run`, everything below is already computed.** You get:

1. published prices of incidents and fines, as a table (section 1),
2. **one chart**: how many UK firms had a breach and what the worst one cost them (section 3),
3. what misses that judged watches have cost **your** firm, from your own records only (section 4),
4. a PDF report, `costs.pdf` (section 5, with a link).

**There is no "judged saves you X" figure.** Nobody has measured that: judged has no customer data. Without your
records section 4 says so and shows no number.

**To add your records:** fill in section 4, then menu **Run → Run All Cells**.
The numbers need only the Python standard library; the chart and the PDF also need matplotlib.

In [ ]:
import json, re, urllib.request
from pathlib import Path

OFFLINE = False          # True: no downloads at all (rates and source pages are skipped)
CURRENCY = "EUR"         # the currency every figure is shown in

DATA = json.loads(Path("costdata.json").read_text())

def download(url, limit=3_000_000, tries=2):
    req = urllib.request.Request(url, headers={"User-Agent": "judged-notebook/0 public list download"})
    for n in range(tries):
        try:
            with urllib.request.urlopen(req, timeout=30) as r:
                return r.read(limit).decode("utf-8", "replace")
        except TimeoutError:
            if n == tries - 1:
                raise

def ecb_rates():
    # {currency: units per 1 EUR}; None when offline or the download fails
    if OFFLINE:
        return None
    try:
        xml = download("https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml")
        rates = {"EUR": 1.0}
        for cur, rate in re.findall(r"currency='([A-Z]{3})'\s+rate='([0-9.]+)'", xml):
            rates[cur] = float(rate)
        return rates if len(rates) > 1 else None
    except (OSError, ValueError) as e:
        print("no exchange rates:", e)
        return None

RATES = ecb_rates()
print("rates:", "downloaded, %d currencies" % len(RATES) if RATES else "none, figures stay in their own currency")

## 1. Published figures in your currency

In [ ]:
def convert(amount, src, dst, rates=None):
    rates = RATES if rates is None else rates
    if src == dst:
        return amount
    if not rates or src not in rates or dst not in rates:
        return None
    return amount / rates[src] * rates[dst]

def show(f, cur=None):
    cur = cur or CURRENCY
    lo, hi = convert(f["low"], f["currency"], cur), convert(f["high"], f["currency"], cur)
    if lo is None:
        lo, hi, cur = f["low"], f["high"], f["currency"]
    txt = f"{cur} {lo:,.0f}" + (f" to {hi:,.0f}" if round(lo) != round(hi) else "")
    per = {"year": 1, "device-year": 1}.get(f["period"])
    if per:
        txt += f"   ({lo/12:,.0f} a month, {lo/365:,.2f} a day)"
    if f["period"] == "minute":
        txt += f"   ({lo*60:,.0f} to {hi*60:,.0f} an hour)"
    return txt

for f in DATA["figures"]:
    print(f"{f['label']:<70} {show(f)}  per {f['period']}  [{f['kind']}: {f['source']}]")
print("\nkind: published = survey or law; vendor = said by someone who sells the fix; computed = arithmetic on a published figure")

## 2. Do the sources still say it? (downloads each page once)

In [ ]:
def verify(figures=DATA["figures"]):
    pages, out = {}, []
    for f in figures:
        if f["url"] not in pages:
            try:
                pages[f["url"]] = download(f["url"])
            except (OSError, ValueError) as e:
                pages[f["url"]] = e
        page = pages[f["url"]]
        if isinstance(page, Exception):
            out.append((f["id"], "unreachable", str(page)[:70]))
        elif not f.get("quote"):
            out.append((f["id"], "page only", "no exact quote (%s figure): open the page and read it yourself" % f["kind"]))
        elif re.sub(r"\s+", " ", f["quote"]) in re.sub(r"\s+", " ", page):
            out.append((f["id"], "found", f["quote"]))
        else:
            out.append((f["id"], "NOT FOUND", f["quote"] + " is gone from the page: the figure may have changed"))
    return out

checked = [] if OFFLINE else verify()
CHECKS = {r[0]: r[1] for r in checked}
for row in checked:
        print("%-12s %-24s %s" % (row[1], row[0], row[2]))

## 3. What firms said their worst breach cost (UK government survey)

Official survey, self-reported costs of the **single most disruptive** breach of the last 12 months, among firms that
noticed one. Read it as: 95 of 100 said it cost that much **or less**. The government stopped publishing the average
this year, so the rare very large cases cannot be turned into a yearly expected cost. These are costs of breaches,
not savings from judged.

In [ ]:
SURVEY = DATA["uk_survey"]

def survey_rows(table="all_answers", cur=None):
    cur = cur or CURRENCY
    if convert(1, SURVEY["currency"], cur) is None:
        cur = SURVEY["currency"]
    return cur, {who: [convert(v, SURVEY["currency"], cur) for v in vals] for who, vals in SURVEY[table].items()}

for table in ("all_answers", "only_above_zero"):
    cur, got = survey_rows(table)
    print(table.replace("_", " ") + ", " + cur)
    print(f"  {'':<18}" + "".join(f"{lv:>18}" for lv in SURVEY["levels"]))
    for who, vals in got.items():
        print(f"  {who:<18}" + "".join(f"{v:>18,.0f}" for v in vals))
print("firms that noticed a breach or attack in 12 months, %:", SURVEY["had_a_breach_percent"])
print(SURVEY["source"])
if not OFFLINE:
    try:
        print("source check:", "found" if SURVEY["quote"] in download(SURVEY["url"]) else "NOT FOUND: the page may have changed")
    except (OSError, ValueError) as e:
        print("source check: unreachable", str(e)[:70])

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("No chart and no PDF: matplotlib is not installed in this Python (pip install matplotlib)")

SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
RAMP = ("#e4e3df", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b")   # grey = nothing happened, then one hue, darker = dearer

def survey_slices(cur=None):
    # [(label, percent of ALL businesses)]: the 43% with a breach split at the published cost percentiles
    cur, got = survey_rows(cur=cur)
    _, p75, p90, p95 = got["all businesses"]
    hit = SURVEY["had_a_breach_percent"]["all"]
    return [("No breach or attack noticed", 100 - hit),
            (f"Worst one cost {cur} {p75:,.0f} or less (many: nothing)", hit * 0.75),
            (f"Worst one cost {cur} {p75:,.0f} to {p90:,.0f}", hit * 0.15),
            (f"Worst one cost {cur} {p90:,.0f} to {p95:,.0f}", hit * 0.05),
            (f"Worst one cost more than {cur} {p95:,.0f}", hit * 0.05)]

def survey_figure(size=(11.69, 6.6)):
    slices = survey_slices()
    hit = SURVEY["had_a_breach_percent"]["all"]
    page = plt.figure(figsize=size, facecolor=SURFACE)
    page.text(0.05, 0.92, "Cyber breaches at UK businesses, last 12 months", fontsize=19, color=INK, weight="bold")
    page.text(0.05, 0.865, "Share of all businesses. Cost = the single most disruptive breach, as the business estimated it.", fontsize=11, color=INK2)
    ax = page.add_axes([0.03, 0.13, 0.40, 0.68])
    ax.pie([s[1] for s in slices], colors=RAMP, startangle=90, counterclock=False,
           wedgeprops={"width": 0.34, "edgecolor": SURFACE, "linewidth": 2})
    ax.text(0, 0.08, f"{hit}%", ha="center", va="center", fontsize=34, color=INK, weight="bold")
    ax.text(0, -0.24, "noticed a breach\nor attack", ha="center", va="center", fontsize=10, color=INK2)
    key = page.add_axes([0.47, 0.13, 0.48, 0.68])
    key.set_xlim(0, 1); key.set_ylim(0, 1); key.axis("off")
    for n, ((label, pct), color) in enumerate(zip(slices, RAMP)):
        y = 0.80 - n * 0.15
        key.add_patch(plt.Rectangle((0.0, y - 0.025), 0.035, 0.05, color=color, linewidth=0))
        key.text(0.065, y, label, va="center", fontsize=11, color=INK)
        key.text(1.0, y, f"{pct:.0f}%" if pct >= 10 else f"{pct:.1f}%", va="center", ha="right", fontsize=13, color=INK, weight="bold")
        key.plot([0, 1], [y - 0.075, y - 0.075], color="#e4e3df", linewidth=0.8)
    page.text(0.05, 0.065, "Source: " + SURVEY["source"] + ".", fontsize=8.5, color=MUTED)
    page.text(0.05, 0.03, f"Shares computed: {hit}% with a breach, split at the 75th, 90th and 95th cost percentiles (approximate). "
              "Costs of breaches, not savings from judged.", fontsize=8.5, color=MUTED)
    return page

if plt:
    survey_figure()
    plt.show()

## 4. Your firm: only what really happened

Write down past misses that a judged check watches, each with a cost you can show: an invoice, a ticket, a bank
statement, a signed contract. No chances, no shares. Set `RECORDS_SINCE` to the day your records start being
complete. An empty list with a start date means a real zero; no start date means no number.

The result is what those misses **cost you**. It is the most judged could have saved on them, if the warning had been
acted on. It is not a promise about next year.

In [ ]:
from datetime import date

RECORDS_SINCE = None     # "2023-01-01"
INCIDENTS = [
    # {"date": "2025-03-02", "check": "expiry", "what": "certificate expired, shop down 3 h", "cost": 0.0, "proof": "ticket or invoice number"},
]

WATCHED = {
    "expiry": "certificate, key, domain or licence ran out",
    "contract": "contract renewed because the notice date was missed",
    "dependency": "incident through a dependency with a published vulnerability",
    "secret": "incident through a secret left in a file or repository",
    "host": "incident through a host setting judged checks (firewall, SSH, updates, encryption)",
    "mail": "spoofed mail that SPF or DMARC settings let through",
}

def measured(incidents, since, today=None):
    # None = no records yet. Otherwise what the recorded misses cost; "per_year" only with a full year of records.
    if since is None:
        if incidents:
            raise ValueError("set RECORDS_SINCE: without it the incidents cannot be turned into a yearly figure")
        return None
    today = today or date.today()
    start = date.fromisoformat(since)
    if start > today:
        raise ValueError("RECORDS_SINCE is in the future")
    by_check = {}
    for n, i in enumerate(incidents, 1):
        if i.get("check") not in WATCHED:
            raise ValueError(f"incident {n}: check must be one of {', '.join(WATCHED)}")
        if not str(i.get("proof", "")).strip():
            raise ValueError(f"incident {n}: no proof, so it does not count: name the invoice, ticket or statement")
        if not start <= date.fromisoformat(i["date"]) <= today:
            raise ValueError(f"incident {n}: date is outside the period of your records")
        if not i["cost"] >= 0:
            raise ValueError(f"incident {n}: cost must be 0 or more")
        by_check[i["check"]] = by_check.get(i["check"], 0) + i["cost"]
    years = (today - start).days / 365.25
    total = sum(by_check.values())
    return {"since": since, "years": years, "count": len(incidents), "by_check": by_check, "total": total,
            "per_year": total / years if (today - start).days >= 365 else None}

def verdict(m, cur):
    if m is None:
        return "NO NUMBER YET: no records entered. Anything printed here without them would be a guess."
    text = f"{m['count']} recorded misses since {m['since']} ({m['years']:.1f} years) cost {cur} {m['total']:,.0f}"
    if m["per_year"] is None:
        return text + ". Less than a year of records: no yearly figure."
    return text + f", that is {cur} {m['per_year']:,.0f} a year."

MEASURED = measured(INCIDENTS, RECORDS_SINCE)
print(verdict(MEASURED, CURRENCY))
for check, cost in sorted((MEASURED or {}).get("by_check", {}).items(), key=lambda g: -g[1]):
    print(f"  {CURRENCY} {cost:>12,.0f}   {WATCHED[check]}")

## 5. PDF report

The chart, your records and the published figures in `costs.pdf`, next to this notebook.

In [ ]:
import os, textwrap

def table_page(pdf, title, header, body, widths, note=""):
    page = plt.figure(figsize=(11.69, 8.27))          # A4, landscape
    ax = page.add_axes([0.03, 0.12, 0.94, 0.76])
    ax.axis("off")
    ax.set_title(title, fontsize=14, loc="left")
    t = ax.table(cellText=body, colLabels=header, colWidths=widths, cellLoc="left", loc="upper left")
    t.auto_set_font_size(False)
    t.set_fontsize(8)
    t.scale(1, 2.2)
    page.text(0.03, 0.05, note, fontsize=8, va="bottom")
    pdf.savefig(page)
    plt.close(page)

def report(path):
    from matplotlib.backends.backend_pdf import PdfPages
    cur = CURRENCY
    m = measured(INCIDENTS, RECORDS_SINCE)
    with PdfPages(path) as pdf:
        page = survey_figure((11.69, 8.27))
        pdf.savefig(page, facecolor=page.get_facecolor())
        plt.close(page)
        body = [[i["date"], textwrap.fill(i["what"], 46), i["check"], f"{i['cost']:,.0f}", textwrap.fill(i["proof"], 28)] for i in INCIDENTS]
        table_page(pdf, f"Your records, {cur}", ["date", "what happened", "check", "cost", "proof"],
                   body or [["-", "no incidents entered", "-", "-", "-"]], [0.10, 0.40, 0.12, 0.10, 0.28], verdict(m, cur))
        body = [[textwrap.fill(f["label"], 44), textwrap.fill(show(f, cur).split("   ")[0], 24), f["period"],
                 textwrap.fill(f"{f['kind']}: {f['source']}", 34), CHECKS.get(f["id"], "not checked")] for f in DATA["figures"]]
        table_page(pdf, "Published figures", ["what", "price", "per", "source", "source check"], body, [0.38, 0.20, 0.09, 0.23, 0.10],
                   "vendor = said by someone who sells the fix; computed = arithmetic on a published figure. Links: costdata.json.")
    return path

if plt:
    PDF = report(Path(os.environ.get("JUDGED_REPORT_DIR", ".")) / "costs.pdf")
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(PDF), result_html_prefix="Open the PDF report: "))
    except ImportError:
        print("PDF report:", PDF)


## What this does not know

* What judged will save next year. That needs a record of what it caught and what each catch would have cost, from real use.
* Hours saved collecting audit evidence: nobody has timed the same job with and without judged.
* Rare very large breaches: the UK survey publishes no average, and the vendor averages mix very different firms.
* Fines and insurance: they depend on the country, the data and the contract. See `docs/costs.md` for the sources and the ceilings.